# Notebook 5: Feature Engineering
**Project**: Loan Approval Prediction & Banking Analytics

---
## 1. Introduction
In this notebook, we engineer features to capture credit-risk profiles and applicant demographics. 

We read the cleaned data from SQLite database, engineer the features, encode our binary target variable, and export the processed data for modeling.

### Engineered Features:
1. **TotalIncome**: `ApplicantIncome + CoapplicantIncome`
2. **IncomeCategory**: Low, Medium, High, Ultra
3. **Loan_to_Income_Ratio**: `(LoanAmount * 1000) / TotalIncome`
4. **EMI_Estimate**: `(LoanAmount * 1000) / Loan_Amount_Term`
5. **ApplicantType**: Solo vs Joint
6. **FamilySize**: `Dependents + Spouse` (spouse count derived from Married status)
7. **RiskCategory**: Low, Medium, High risk indicators


In [1]:
import pandas as pd
import numpy as np
import os
import sqlite3

# Load data from database
db_path = os.path.join("..", "SQL", "loan_database.db")
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM loan_cleaned", conn)
conn.close()

print(f"Ingested cleaned dataset of shape: {df.shape}")

Ingested cleaned dataset of shape: (614, 13)


## 2. Feature Creation

In [2]:
# 1. Total Income
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']

# 2. Income Category
def get_income_category(income):
    if income < 3500:
        return 'Low'
    elif income < 6000:
        return 'Medium'
    elif income < 10000:
        return 'High'
    else:
        return 'Ultra'

df['IncomeCategory'] = df['TotalIncome'].apply(get_income_category)

# 3. Loan-to-Income Ratio
df['Loan_to_Income_Ratio'] = (df['LoanAmount'] * 1000) / df['TotalIncome']

# 4. EMI Estimate (Approximated simple installment)
df['EMI_Estimate'] = (df['LoanAmount'] * 1000) / df['Loan_Amount_Term']

# 5. Applicant Type (Solo vs Joint application)
df['ApplicantType'] = df['CoapplicantIncome'].apply(lambda x: 'Solo' if x == 0 else 'Joint')

# 6. Family Size (Applicant + Spouse + Dependents)
# Standardize Dependents: Convert '3+' to 3 and cast to int
df['Dependents_Numeric'] = df['Dependents'].apply(lambda x: 3 if x == '3+' else int(x))
df['Spouse_Numeric'] = df['Married'].apply(lambda x: 1 if x == 'Yes' else 0)
df['FamilySize'] = df['Dependents_Numeric'] + df['Spouse_Numeric'] + 1 # Include applicant

# 7. Risk Category (Rule-based based on credit history and income)
def get_risk_category(row):
    if row['Credit_History'] == 0:
        return 'High'
    elif row['TotalIncome'] < 4000:
        return 'Medium'
    else:
        return 'Low'

df['RiskCategory'] = df.apply(get_risk_category, axis=1)

# Drop intermediary columns used for calculation
df.drop(['Dependents_Numeric', 'Spouse_Numeric'], axis=1, inplace=True)

df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,TotalIncome,IncomeCategory,Loan_to_Income_Ratio,EMI_Estimate,ApplicantType,FamilySize,RiskCategory
0,LP001002,Male,No,0,Graduate,No,5849,0.0,128.0,360,1,Urban,Y,5849.0,Medium,21.884083,355.555556,Solo,1,Low
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360,1,Rural,N,6091.0,High,21.014612,355.555556,Joint,3,Low
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360,1,Urban,Y,3000.0,Low,22.000000,183.333333,Solo,2,Medium
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360,1,Urban,Y,4941.0,Medium,24.286582,333.333333,Joint,2,Low
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360,1,Urban,Y,6000.0,High,23.500000,391.666667,Solo,1,Low


## 3. Encode Target Variable

In [3]:
# Encode target: Y=1, N=0
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})
df['Loan_Status'].value_counts()

Loan_Status
1    422
0    192
Name: count, dtype: int64

## 4. Final Review of Correlations

In [4]:
# Verify correlations of new features with the target variable
numeric_df = df.select_dtypes(include=[np.number])
print(numeric_df.corr()['Loan_Status'].sort_values(ascending=False))

Loan_Status             1.000000
Credit_History          0.540556
FamilySize              0.042963
ApplicantIncome        -0.004710
EMI_Estimate           -0.011757
Loan_Amount_Term       -0.022549
TotalIncome            -0.031271
LoanAmount             -0.033214
CoapplicantIncome      -0.059187
Loan_to_Income_Ratio   -0.086141
Name: Loan_Status, dtype: float64


## 5. Export Processed Dataset

In [5]:
# Export to processed folder
processed_path = os.path.join("..", "Dataset", "processed", "loan_processed.csv")
df.to_csv(processed_path, index=False)
print(f"Processed dataset saved successfully to: {processed_path}")

Processed dataset saved successfully to: ..\Dataset\processed\loan_processed.csv
